In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc samplomatic
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Avvio rapido con Executor

<Accordion>
<AccordionItem title="Versioni dei pacchetti">

Il codice in questa pagina è stato sviluppato con i seguenti requisiti.
Ti consigliamo di utilizzare queste versioni o versioni più recenti.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
samplomatic~=0.18.0
```
</AccordionItem>
</Accordion>

Analogamente alla primitiva [Sampler](/guides/get-started-with-sampler), Executor campiona i registri di output dalle esecuzioni di Circuit quantistici, ma non dispone di alcuna soppressione o mitigazione degli errori integrata. È invece parte del [modello di esecuzione diretto](/guides/directed-execution-model) che fornisce gli ingredienti per catturare gli intenti di progettazione sul lato client e spostare la costosa generazione di varianti di Circuit sul lato server. Executor segue le direttive fornite nelle annotazioni e nelle opzioni del circuito, genera e associa i valori dei parametri, esegue i circuito associati sull'hardware e restituisce i risultati dell'esecuzione e i metadati. Non prende alcuna decisione implicita per te e ti offre pieno controllo e trasparenza.

> **Note:** Il pacchetto Qiskit non dispone ancora di una classe base per la primitiva Executor.

## Prima di iniziare
Alcuni degli esempi di codice in questa pagina utilizzano `samplex`, che fa parte del pacchetto Samplomatic. Pertanto, prima di eseguire quei blocchi di codice, devi installare Samplomatic, come mostrato nel seguente blocco di codice. Per ulteriori informazioni, consulta la [documentazione di Samplomatic](https://qiskit.github.io/samplomatic).

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService, Executor
from qiskit_ibm_runtime.quantum_program import QuantumProgram
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from samplomatic.transpiler import generate_boxing_pass_manager
from samplomatic import build

# Initialize the service and choose a backend
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

In [2]:
print(backend)

<IBMBackend('ibm_fez')>


### 2. Create and transpile a circuit

You need at least one circuit to use the Executor primitive.  It can optionally have parameters.

In [3]:
# Generate the circuit
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.h(1)
circuit.cz(0, 1)
circuit.h(1)

# Using `measure_all` automatically creates the necessary
# classical registers.
circuit.measure_all()

The circuit needs to be transformed to only use instructions supported by the QPU (referred to as *instruction set architecture (ISA)* circuits). Use the transpiler to do this.

In [4]:
# Transpile the circuit
preset_pass_manager = generate_preset_pass_manager(
    backend=backend, optimization_level=0
)
isa_circuit = preset_pass_manager.run(circuit)

### 2. Crea e transpila un circuito
Hai bisogno di almeno un circuito per utilizzare la primitiva Executor. Può facoltativamente avere parametri.

In [5]:
# Initialize an empty program
program = QuantumProgram(shots=25)

# Append the circuit to the program
program.append_circuit_item(isa_circuit)

Il circuito deve essere trasformato per usare solo le istruzioni supportate dalla QPU (chiamati circuiti *instruction set architecture (ISA)*). Usa il Transpiler per farlo.

In [6]:
# Generate a boxing pass manager to group gates
# and measurements into boxes and add
# a`Twirl` annotation.
boxes_pm = generate_boxing_pass_manager(
    # Add gate twirling
    enable_gates=True,
    # Add measurement twirling
    enable_measures=True,
)

boxed_circuit = boxes_pm.run(isa_circuit)
boxed_circuit.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/get-started-with-executor/extracted-outputs/245a4574-3ce9-4f77-98c8-af32cde8ac01-0.svg" alt="Output of the previous code cell" />

### 3. Inizializza un `QuantumProgram`
Inizializza un `QuantumProgram` con il tuo carico di lavoro. Un `QuantumProgram` è composto da `QuantumProgramItems`. Tipicamente, ogni elemento è composto da un circuito, un insieme di valori dei parametri e possibilmente un `samplex` per randomizzare il contenuto del circuito. Per i dettagli completi, consulta [Input e output di Executor](/guides/executor-input-output).

La seguente cella inizializza un `QuantumProgram` e specifica di eseguire 25 shot. Poi aggiunge il circuito target transpilato.

In [7]:
# Build the template circuit and the samplex
template_circuit, samplex = build(boxed_circuit)

# Append the template circuit and samplex as a `samplex_item`
program.append_samplex_item(
    template_circuit,
    samplex=samplex,
    shape=(num_randomizations := 20,),
)

### 4. Opzionale: Raggruppa gate e misurazioni in box annotati
Raggruppare le istruzioni in box e annotarle è il modo principale per specificare il tuo intento. Nell'esempio seguente, usiamo `generate_boxing_pass_manager` e i suoi parametri di twirling per raggruppare i gate a due qubit e le misurazioni in box e applicare l'annotazione di twirling.

In [8]:
# Initialize an Executor with the default options
executor = Executor(mode=backend)

# Submit the job
job = executor.run(program)
job

<RuntimeJobV2('d8286580bvlc73d1vmsg', 'executor')>

In [9]:
# Retrieve the result
result = job.result()

### 6. Invoca Executor e ottieni i risultati
Esegui il `QuantumProgram` su un backend IBM&reg; utilizzando la primitiva `Executor` con le opzioni predefinite. Consulta le [opzioni di Executor](/guides/executor-options) per informazioni sulle opzioni disponibili.